<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0;">Lab 01: LangGraph Environment Setup &amp; Validation</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">Install the LangGraph SDK and understand graph-based agent architecture</p>
</div>

**What We Learn:** How to install the LangGraph SDK, understand the graph-based agent architecture (nodes, edges, state), and validate your environment for building declarative agents.

---


## 🧳 The Contoso Travel Story

Contoso Travel has seen prompted agents and MAF agents. Now they want to explore a declarative, graph-based approach where the agent's control flow is defined as a visual graph — making it easier to reason about, test, and modify complex multi-step workflows.

- **The Problem:** Complex agent workflows with branching logic, retries, and conditional routing are hard to express imperatively. The team wants a declarative approach that makes the control flow explicit and visual.
- **This Lab Solves:** Setting up LangGraph with the Foundry hosting adapter so the team can build graph-based agents that are easy to visualize and modify.


## 1. Install LangGraph Dependencies

We install the LangGraph SDK, the Foundry hosting adapter, LangChain core libraries, and authentication packages.

---


In [ ]:
%pip install azure-ai-agentserver-langgraph==1.0.0b12 langgraph langchain-core langchain-openai azure-ai-projects>=2.0.0 azure-identity python-dotenv --quiet

## 2. Load Environment & Connect

Load credentials from the shared <code>.env</code> file and connect to the Foundry project.

---


In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))
endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4.1-mini")

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
print(f"✅ Connected to Foundry project")
print(f"   Model: {model_name}")

## 3. Understand the LangGraph Architecture

LangGraph models agents as **state graphs** — directed graphs where nodes are processing steps and edges define control flow. This makes complex workflows explicit and visual.

---

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #1565c0; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
<b>StateGraph</b> — The graph definition object. You add nodes (Python functions) and edges (connections) to build the agent's control flow.
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #1565c0; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
<b>MessagesState</b> — A built-in state type that manages conversation message history. Each node reads from and writes to this shared state.
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #1565c0; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
<b>Nodes</b> — Python functions that process the current state and return updates. Each node performs one step of the workflow (e.g., call the LLM, execute a tool).
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #1565c0; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
<b>Edges</b> — Connections between nodes. Can be <b>static</b> (always follow this path) or <b>conditional</b> (choose path based on state).
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #1565c0; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
<b>START / END</b> — Special sentinel nodes that mark the graph's entry and exit points. Every graph begins at START and terminates at END.
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #1565c0; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
<b><code>compile()</code></b> — Converts the graph definition into a runnable agent. The compiled graph can be invoked like a function.
</div>

The classic tool-calling agent follows this pattern:

```
START → llm_call → should_continue? → tools → llm_call (loop)
                                    → END
```


## 4. Understand the Hosting Adapter

The Foundry hosting adapter wraps a compiled LangGraph agent so it can be served as a hosted agent.

---

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2e8b57; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
<b><code>from_langgraph(compiled_graph)</code></b> wraps the compiled graph for Foundry hosting. This is the same pattern as MAF's <code>from_agent_framework()</code> — the adapter translates between the Responses API and your agent's native interface.
</div>

```python
from azure.ai.agentserver.langgraph import from_langgraph

# Wrap and serve
from_langgraph(agent).run()  # Starts HTTP server on port 8088
```


## 5. Three Approaches Compared

How does LangGraph fit alongside the other agent paradigms we've explored?

---

| Aspect | Prompted | MAF | LangGraph |
|--------|----------|-----|----------|
| Paradigm | Declarative (instructions) | Imperative (Python classes) | Declarative (graph) |
| Control flow | Foundry-managed | Your code | Graph edges |
| Tool handling | FunctionTool + external calls | In-process functions | @tool + tool nodes |
| Visualization | N/A | Code inspection | Graph visualization |
| Best for | Simple agents | Complex business logic | Multi-step workflows |


## 6. Summary & Next Steps

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2e8b57; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
<b>✅ What We Accomplished</b><br/>Installed LangGraph and all dependencies, connected to the Foundry project, and understood the graph-based architecture — nodes, edges, state, and the hosting adapter.
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #1565c0; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
<b>💡 Key Takeaway</b><br/>LangGraph makes agent control flow <b>explicit and visual</b>. Instead of burying logic in Python methods, you define a graph where each step is a node and each transition is an edge.
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #ff9800; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;">
<b>➡️ Next:</b> In <a href="lab-02-agent.ipynb">Lab 02</a>, we'll build our first LangGraph agent — defining the concierge as a state graph with an LLM node and compiling it into a runnable agent.
</div>


In [ ]:
# Cleanup: close clients
project_client.close()
credential.close()
print("✅ Clients closed. Setup complete!")